# Generating Training data

Quantum version has state rpeparation circuits

Classical version has classical feature vectors (derived from state vectors).

The KNN will be trined on this data to classify the entanglement and we will evaluate the performance. 

## W state

In [2]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate, XGate, CXGate

import numpy as np

# 1-qubit G gate
def G_gate(p):
    mat = np.array([[np.sqrt(p), -np.sqrt(1-p)],
                    [np.sqrt(1-p), np.sqrt(p)]], dtype=complex)
    return UnitaryGate(mat, label=f"G({p})")

class WCircuitLinearQC:
    def __init__(self, n):
        self.n = n
        self.circuit = QuantumCircuit(n)
        
        # X on first qubit
        self.circuit.x(0)
        
        # Define B sequence
        def B(p, q1, q2):
            # Apply G to the second qubit (matches original W construction)
            self.circuit.append(G_gate(1/p), [q2])
            # SWAP, CNOT, SWAP
            self.circuit.swap(q1, q2)
            self.circuit.cx(q1, q2)
            self.circuit.swap(q1, q2)
        
        # Apply B along the chain
        for i in range(n-1):
            B(n-i, i, i+1)

    def get_circuit(self):
        return self.circuit

# Example: 3-qubit W state
w_linear = WCircuitLinearQC(3)
print(w_linear.get_circuit())


               ┌───┐                                
q_0: ──────────┤ X ├───────────X───■───X────────────
     ┌─────────┴───┴─────────┐ │ ┌─┴─┐ │            
q_1: ┤ G(0.3333333333333333) ├─X─┤ X ├─X──X───■───X─
     └───────┬────────┬──────┘   └───┘    │ ┌─┴─┐ │ 
q_2: ────────┤ G(0.5) ├───────────────────X─┤ X ├─X─
             └────────┘                     └───┘   


## GHZ State

In [3]:
class GHZCircuitQC:
    def __init__(self, n):
        self.n = n
        self.circuit = QuantumCircuit(n)
        self.circuit.h(0)
        for i in range(n-1):
            self.circuit.cx(i, i+1)
    
    def get_circuit(self):
        return self.circuit

# Example usage
ghz = GHZCircuitQC(3)
print(ghz.get_circuit())


     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘


## QFT State

In [14]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT

class QFTCircuitQC:
    def __init__(self, n):
        self.n = n
        self.circuit = QFT(n, do_swaps=True).decompose()  # Create QFT circuit

    def get_circuit(self):
        return self.circuit

# Example usage
qft = QFTCircuitQC(3)
print(qft.get_circuit())

                                          ┌───┐   
q_0: ────────────────────■────────■───────┤ H ├─X─
                   ┌───┐ │        │P(π/2) └───┘ │ 
q_1: ──────■───────┤ H ├─┼────────■─────────────┼─
     ┌───┐ │P(π/2) └───┘ │P(π/4)                │ 
q_2: ┤ H ├─■─────────────■──────────────────────X─
     └───┘                                        


C:\Users\aadik\AppData\Local\Temp\ipykernel_23344\4110475288.py:7: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  self.circuit = QFT(n, do_swaps=True).decompose()  # Create QFT circuit


## Random State

In [13]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit

class RandomCircuitQC:
    def __init__(self, n, depth, max_operands=2, seed=None):
        """
        n: number of qubits
        depth: number of layers
        max_operands: maximum qubits a gate acts on
        seed: random seed for reproducibility
        """
        self.n = n
        self.depth = depth
        self.circuit = random_circuit(n, depth, max_operands=max_operands, seed=seed)

    def get_circuit(self):
        return self.circuit

# Example usage
rand_circ = RandomCircuitQC(3, 5, seed=42)
print(rand_circ.get_circuit())


     ┌────────────────────────┐┌──────────────────────────┐              »
q_0: ┤0                       ├┤ U(1.4278,3.4846,0.40098) ├──────■───────»
     │                        │└─────────┬─────┬──────────┘┌─────┴──────┐»
q_1: ┤  (XX-YY)(4.7824,4.939) ├──────────┤ Sdg ├───────────┤ Rz(6.0991) ├»
     │                        │┌─────────┴─────┴──────────┐└────────────┘»
q_2: ┤1                       ├┤ U3(4.0455,5.1696,2.7861) ├──────────────»
     └────────────────────────┘└──────────────────────────┘              »
«      ┌─────────────┐      
«q_0: ─┤ Rx(0.96943) ├───X──
«     ┌┴─────────────┴┐  │  
«q_1: ┤0              ├──X──
«     │  Rxx(0.27523) │┌───┐
«q_2: ┤1              ├┤ T ├
«     └───────────────┘└───┘


# Entanglement Entropy

In [12]:
from qiskit.quantum_info import Statevector
def cut_vn_entropies(statevector):
    n = int(np.log2(len(statevector)))
    entropies = []
    for k in range(1, n):
        sv_k = statevector.reshape(2**k, 2**(n-k))
        s = np.linalg.svd(sv_k, compute_uv=False)
        lam = s**2
        lam = np.clip(lam, 1e-15, 1.0)
        entropies.append(-np.sum(lam * np.log2(lam)))
    return np.array(entropies)



# --- Build 3-qubit W-linear circuit ---
n_qubits = 3
wc = WCircuitLinearQC(n_qubits)
qc = wc.get_circuit()
print(qc.draw())

print("circuit prepared state vector")
# --- Get statevector ---
sv = Statevector.from_instruction(qc).data
print(sv)

print("entanglement entropy")
print(cut_vn_entropies(sv))

def avg_entanglement(circ):
    sv = Statevector.from_instruction(qc).data
    return np.mean(cut_vn_entropies(sv))

print(avg_entanglement(qc))

               ┌───┐                                
q_0: ──────────┤ X ├───────────X───■───X────────────
     ┌─────────┴───┴─────────┐ │ ┌─┴─┐ │            
q_1: ┤ G(0.3333333333333333) ├─X─┤ X ├─X──X───■───X─
     └───────┬────────┬──────┘   └───┘    │ ┌─┴─┐ │ 
q_2: ────────┤ G(0.5) ├───────────────────X─┤ X ├─X─
             └────────┘                     └───┘   
circuit prepared state vector
[0.        +0.j 0.40824829+0.j 0.57735027+0.j 0.        +0.j
 0.57735027+0.j 0.        +0.j 0.        +0.j 0.40824829+0.j]
entanglement entropy
[1.         0.91829583]
0.9591479170272446
